# Module B — Gaze Zone v3 (WHENet + L2CS-Net Fusion)

## Why we are NOT retraining EfficientNet again

Fine-tune v2 achieved Forward F1 = 0.108 despite using 99,349 LISA frames + NTHU pseudo-labels.
The root cause is **not fixable by more training**:
- NTHU val/test has Forward = 8.8% of frames (drivers are intentionally distracted)
- A model always predicting 'not Forward' achieves 91.2% accuracy — optimal for NTHU
- EfficientNet learned this pattern across 25 training epochs
- Running v3 would give Forward F1 ≈ 0.12–0.18 after another 6 hours of compute

## What we do instead: pure geometry, zero training

**WHENet** (head pose, ±3-5° error, yaw ±120°) tells us where the head is pointing.  
**L2CS-Net** (gaze direction, trained on Gaze360 360°) tells us where the eyes are pointing.  
**Fused gaze** = 35% WHENet + 65% L2CS-Net → mapped to zone via `pose_to_zone()`.  
**A-pillar calibration**: first 5 seconds driver looks forward → measures camera offset angle.

Expected performance:
| Method | Forward F1 | Zone macro F1 |
|---|---|---|
| EfficientNet v2 (retrained) | 0.108 | 0.196 |
| WHENet only | ~0.55-0.65 | ~0.45-0.55 |
| L2CS-Net only | ~0.65-0.75 | ~0.55-0.65 |
| **WHENet + L2CS-Net fused** | **~0.70-0.80** | **~0.60-0.70** |

## Datasets required in Kaggle Data tab
1. `suyashpokle2/nthu-ddd` — NTHU-DDD frames for evaluation
2. `suyashpokle2/manifests-cv-kaggle-fullpaths` — fold manifests
3. `suyashpokle2/lisa-gaze-v2` — LISA frames (for per-zone sanity check)

## Outputs
- `gaze_zone_predictor.py` — standalone inference class for Streamlit
- `b_gaze_v3_meta.json` — zone thresholds, calibration params
- `l2csnet_gaze360.onnx` — L2CS-Net ONNX export (for Snowflake)
- Upload all to `@DEMO_DB.PUBLIC.DRIVER_SAFETY_MODELS`

In [1]:
# import subprocess, sys
# def pip(*args): subprocess.check_call([sys.executable,"-m","pip","install","-q",*args])
# # l2cs: official L2CS-Net package (PyTorch, ResNet50 backbone, Gaze360 weights)
# pip("l2cs", "onnx", "onnxruntime", "opencv-python-headless>=4.8")
# print("Packages ready.")

In [2]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/edavalosanaya/L2CS-Net.git@main")
pip("onnx", "onnxruntime", "opencv-python-headless>=4.8")
print("Packages ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 84.5 MB/s eta 0:00:00
Packages ready.


In [3]:
import gc, json, math, os, shutil, urllib.request, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

import torch
import onnxruntime as ort

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", DEVICE)

torch: 2.10.0+cu128 | device: cuda


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# Paths
NTHU_DATA_DIR  = "/kaggle/input/datasets/suyashpokle2/nthu-ddd/NTHU DDD"
LISA_DATA_DIR  = "/kaggle/input/datasets/suyashpokle2/lisa-gaze-v2/lisa_gaze_v2"
MANIFEST_ROOT  = "/kaggle/input/datasets/suyashpokle2/manifests-cv-kaggle-fullpaths/manifests_cv_kaggle_fullpaths"
OUTPUT_DIR     = "/kaggle/working/gaze_v3"
YUNET_PATH     = "/kaggle/working/face_detection_yunet_2023mar.onnx"
#WHENET_PATH    = "/kaggle/working/whenet_1x3x224x224_prepost.onnx"  # user already has this
L2CS_ONNX_OUT  = "/kaggle/working/l2csnet_gaze360.onnx"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Zone definitions
ZONE_CLASSES = ["Forward","Lap","Left Mirror","Radio",
                "Rearview","Right Mirror","Shoulder","Speedometer"]
ZONE_TO_IDX  = {z:i for i,z in enumerate(ZONE_CLASSES)}
IDX_TO_ZONE  = {i:z for i,z in enumerate(ZONE_CLASSES)}

# ── Gaze-to-zone angle thresholds (degrees, camera frame)
# These match the canonical LISA zone geometry.
# pose_to_zone() receives WORLD-FRAME gaze (after camera offset correction).
# Camera offset is calibrated per session (first 5s driver looks forward).
# CORRECTED (frontal + A-pillar 0-30° geometry)
ZONE_THRESHOLDS = {
    "yaw_shoulder":    35.0,
    "yaw_mirror":      20.0,
    "pitch_lap":      -25.0,
    "pitch_down":     -15.0,
    "yaw_radio":       15.0,
    "pitch_rearview":  15.0,
    "yaw_forward":     15.0,
    "pitch_forward":   15.0,
}
# ── Fusion weights
# L2CS-Net (eye gaze) gets higher weight than WHENet (head pose).
# At high face yaw (>30°), eye gaze becomes unreliable → fall back to WHENet.
FUSION_W_WHENET = 0.35
FUSION_W_L2CS   = 0.65
HIGH_YAW_FALLBACK_DEG = 35.0  # face yaw above this → use WHENet only

# ── Temporal smoothing (applied to fused gaze angles before zone mapping)
TEMPORAL_SMOOTH_FRAMES = 3

print("Config ready.")
print(f"Zone classes: {ZONE_CLASSES}")
print(f"Fusion: WHENet×{FUSION_W_WHENET} + L2CS×{FUSION_W_L2CS}")

Config ready.
Zone classes: ['Forward', 'Lap', 'Left Mirror', 'Radio', 'Rearview', 'Right Mirror', 'Shoulder', 'Speedometer']
Fusion: WHENet×0.35 + L2CS×0.65


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# YUNET FACE DETECTOR
# ══════════════════════════════════════════════════════════════════════════════

def download_yunet(dst):
    if os.path.exists(dst): print("YuNet present."); return
    url = ("https://github.com/opencv/opencv_zoo/raw/main/"
           "models/face_detection_yunet/face_detection_yunet_2023mar.onnx")
    print("Downloading YuNet…")
    urllib.request.urlretrieve(url, dst)
    print("Done.")

download_yunet(YUNET_PATH)
_yunet = cv2.FaceDetectorYN.create(
    model=YUNET_PATH, config="", input_size=(320,320),
    score_threshold=0.50, nms_threshold=0.30, top_k=1)

def yunet_detect(frame_bgr):
    """Returns (face_224, face_yaw_approx, det_row) or (None, 0, None)."""
    h,w = frame_bgr.shape[:2]
    _yunet.setInputSize((w,h))
    _,faces = _yunet.detect(frame_bgr)
    if faces is None or len(faces)==0: return None, 0.0, None
    det = faces[0]
    bx,by,bw,bh = int(det[0]),int(det[1]),int(det[2]),int(det[3])
    score = float(det[14]) if len(det)>14 else 0.0
    if score < 0.40: return None, 0.0, None
    # 30% margin crop → 224×224 for WHENet
    mx,my = int(bw*0.30),int(bh*0.30)
    x1=max(0,bx-mx); y1=max(0,by-my)
    x2=min(w,bx+bw+mx); y2=min(h,by+bh+my)
    if x2<=x1 or y2<=y1: return None, 0.0, None
    face_224 = cv2.resize(frame_bgr[y1:y2,x1:x2],(224,224),interpolation=cv2.INTER_LINEAR)
    # Approximate face yaw from eye positions (for fusion weight decision)
    re_x,le_x = float(det[4]),float(det[6])
    iod = abs(le_x-re_x)
    face_w_px = bw
    # Rough yaw estimate: if IOD << face_width, face is turned away
    face_yaw_approx = float(np.degrees(np.arccos(np.clip(iod/max(face_w_px*0.45,1),-1,1))))
    return face_224, face_yaw_approx, det

print("YuNet detector ready.")

YuNet present.
YuNet detector ready.


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# WHENet — HEAD POSE ESTIMATION (ONNX)
# Input:  (1, 3, 224, 224) RGB, ImageNet normalised
# Output: [yaw, pitch, roll] in degrees  (yaw ±120°, pitch ±90°)
#
# User has already uploaded whenet_1x3x224x224_prepost.onnx to Snowflake.
# For Kaggle, we need it locally. Download from GitHub if not present.
# ══════════════════════════════════════════════════════════════════════════════

# def download_whenet(dst):
#     if os.path.exists(dst): print("WHENet present."); return
#     # Try Kaggle dataset path first (if user uploaded it)
#     candidates = [
#         "/kaggle/input/mod-b-pretrained/whenet_1x3x224x224_prepost.onnx",
#         "/kaggle/input/whenet/whenet_1x3x224x224_prepost.onnx",
#     ]
#     for c in candidates:
#         if os.path.exists(c): shutil.copy2(c, dst); print(f"Copied WHENet from {c}"); return
#     # GitHub release
#     url = ("https://github.com/Ascend-Research/HeadPoseEstimation-WHENet/"
#            "releases/download/v1.0/whenet_1x3x224x224_prepost.onnx")
#     print("Downloading WHENet ONNX (~23 MB)…")
#     urllib.request.urlretrieve(url, dst)
#     print("Done.")

# download_whenet(WHENET_PATH)

import os, shutil, urllib.request
import onnxruntime as ort
import numpy as np
import cv2

WHENET_PATH = "/kaggle/working/HeadPoseEstimation-WHENet-yolov4-onnx-openvino/saved_model_224x224/whenet_1x3x224x224_prepost.onnx"

def download_whenet(dst):
    if os.path.exists(dst):
        print("WHENet present at:", dst)
        return

    os.makedirs(os.path.dirname(dst), exist_ok=True)

    url = "https://github.com/PINTO0309/HeadPoseEstimation-WHENet-yolov4-onnx-openvino/releases/download/v1.0.4/whenet_1x3x224x224_prepost.onnx"
    print("Downloading WHENet ONNX…")
    urllib.request.urlretrieve(url, dst)
    print("Done:", dst)

download_whenet(WHENET_PATH)

opts = ort.SessionOptions()
opts.intra_op_num_threads = 2
sess = ort.InferenceSession(
    WHENET_PATH,
    sess_options=opts,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)

print("Loaded WHENet successfully")

_whenet_opts = ort.SessionOptions()
_whenet_opts.intra_op_num_threads = 2
_whenet_sess = ort.InferenceSession(WHENET_PATH, _whenet_opts,
                                     providers=["CUDAExecutionProvider","CPUExecutionProvider"])
_NORM_MEAN = np.array([0.485,0.456,0.406],dtype=np.float32)
_NORM_STD  = np.array([0.229,0.224,0.225],dtype=np.float32)

def whenet_predict(face_bgr_224):
    """Returns (yaw, pitch, roll) in degrees or (None,None,None)."""
    if face_bgr_224 is None: return None,None,None
    try:
        img = cv2.cvtColor(face_bgr_224, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        img = (img-_NORM_MEAN)/_NORM_STD
        inp = img.transpose(2,0,1)[np.newaxis].astype(np.float32)
        out = _whenet_sess.run(None, {_whenet_sess.get_inputs()[0].name: inp})
        # WHENet output format varies by export:
        # Format A: single output [yaw, pitch, roll]
        # Format B: three outputs [yaw], [pitch], [roll]
        if len(out)==3:
            yaw,pitch,roll = float(out[0].squeeze()),float(out[1].squeeze()),float(out[2].squeeze())
        else:
            angles = out[0].squeeze()
            yaw,pitch,roll = float(angles[0]),float(angles[1]),float(angles[2])
        return yaw,pitch,roll
    except Exception as e:
        return None,None,None

# Sanity check
dummy = np.zeros((224,224,3),dtype=np.uint8)
yaw_t,pitch_t,roll_t = whenet_predict(dummy)
print(f"WHENet ready. Test output: yaw={yaw_t:.1f}° pitch={pitch_t:.1f}° roll={roll_t:.1f}°")

WHENet present at: /kaggle/working/HeadPoseEstimation-WHENet-yolov4-onnx-openvino/saved_model_224x224/whenet_1x3x224x224_prepost.onnx
Loaded WHENet successfully
WHENet ready. Test output: yaw=-32.9° pitch=0.3° roll=-21.7°


In [7]:
# !mkdir -p /kaggle/working/L2CS-Net/models
# !wget https://github.com/edavalosanaya/L2CS-Net/releases/download/v0.1/L2CSNet_gaze360.pkl \
#      -O /kaggle/working/L2CS-Net/models/L2CSNet_gaze360.pkl

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# L2CS-Net — GAZE ESTIMATION
#
# Strategy:
#   1. Use pip install l2cs (official PyTorch) for this notebook
#   2. Then export to ONNX for Snowflake deployment
#
# L2CS-Net processes 448×448 face crops and outputs (yaw, pitch) gaze
# in degrees relative to the camera axis.
# ══════════════════════════════════════════════════════════════════════════════

import torch
from l2cs import Pipeline, render
from l2cs.utils import select_device

# L2CS-Net pipeline: automatically downloads Gaze360 weights (~90 MB, first run only)
#_l2cs_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# _l2cs_pipe = Pipeline(
#     weights="Gaze360",   # best weights for wide-angle gaze
#     arch="ResNet50",
#     device=_l2cs_device,
#     include_detector=False,  # we use YuNet, not RetinaFace
#     confidence_threshold=0.40,
# )


import os
import torch
from l2cs import Pipeline

weights_path = "/kaggle/input/models/suyashpokle/l2cs-weights/pytorch/default/1/L2CSNet_gaze360.pkl"
assert os.path.exists(weights_path), f"Missing: {weights_path}"

_l2cs_pipe = Pipeline(
    weights=weights_path,
    arch="ResNet50",
    device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),
    include_detector=False,
    confidence_threshold=0.40,
)

def l2cs_predict(face_bgr):
    """
    Predict gaze (yaw, pitch) from a face crop.
    L2CS-Net expects the face region — we pass our YuNet face crop.
    Returns (yaw_deg, pitch_deg) or (None, None).
    yaw:   positive = right, negative = left
    pitch: positive = up, negative = down
    """
    if face_bgr is None: return None, None
    try:
        # L2CS-Net expects faces detected in the image.
        # We pass a face-only crop as if it were a full frame.
        # Resize to 224×224 first for speed (L2CS-Net resizes internally)
        face_rgb = cv2.cvtColor(
            cv2.resize(face_bgr,(224,224),interpolation=cv2.INTER_LINEAR),
            cv2.COLOR_BGR2RGB
        )
        # L2CS-Net .step() expects BGR numpy (H,W,3)
        results = _l2cs_pipe.step(face_bgr)
        if results is None: return None, None
        pitch_arr = results.pitch
        yaw_arr   = results.yaw
        if pitch_arr is None or len(pitch_arr)==0: return None, None
        # Take first face result
        yaw   = float(np.degrees(yaw_arr[0]))   if yaw_arr is not None else None
        pitch = float(np.degrees(pitch_arr[0])) if pitch_arr is not None else None
        # L2CS outputs in radians — convert if needed
        # (some versions output degrees already, some radians — check magnitude)
        if yaw is not None and abs(yaw) < 3.5:  # likely radians
            yaw   = float(np.degrees(yaw_arr[0]))
            pitch = float(np.degrees(pitch_arr[0]))
        return yaw, pitch
    except Exception as e:
        return None, None

# Sanity check
dummy_face = np.zeros((224,224,3),dtype=np.uint8)
y_t,p_t = l2cs_predict(dummy_face)
print(f"L2CS-Net ready. Test output: yaw={y_t} pitch={p_t}")

L2CS-Net ready. Test output: yaw=None pitch=None


In [9]:
# !pip install onnxscript

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPORT L2CS-Net TO ONNX (for Snowflake deployment)
#
# This cell exports L2CS-Net backbone to ONNX so it can run via onnxruntime
# in Snowflake (no PyTorch needed at inference time).
# ══════════════════════════════════════════════════════════════════════════════

def export_l2cs_onnx(pipeline, out_path, img_size=448):
    """Export L2CS-Net backbone to ONNX. Input: (1,3,img_size,img_size)."""
    import torch
    model = pipeline.model.cpu().eval()
    dummy = torch.zeros(1, 3, img_size, img_size)
    torch.onnx.export(
        model, dummy, out_path,
        input_names=["face_input"],
        output_names=["pitch_pred","yaw_pred"],
        opset_version=11,
        do_constant_folding=True,
        dynamic_axes={"face_input":{0:"batch"}},
    )
    print(f"L2CS-Net ONNX exported: {out_path} ({os.path.getsize(out_path)//1024//1024} MB)")
    model.to(_l2cs_device)  # move back

if not os.path.exists(L2CS_ONNX_OUT):
    export_l2cs_onnx(_l2cs_pipe, L2CS_ONNX_OUT)
else:
    print(f"L2CS-Net ONNX already exists: {L2CS_ONNX_OUT}")

# Verify ONNX session
_l2cs_ort_opts = ort.SessionOptions()
_l2cs_ort_opts.intra_op_num_threads = 2
_l2cs_ort_sess = ort.InferenceSession(L2CS_ONNX_OUT, _l2cs_ort_opts,
                                        providers=["CPUExecutionProvider"])
print(f"L2CS-Net ONNX session ready.")
print(f"  Inputs:  {[i.name for i in _l2cs_ort_sess.get_inputs()]}")
print(f"  Outputs: {[o.name for o in _l2cs_ort_sess.get_outputs()]}")

L2CS-Net ONNX already exists: /kaggle/working/l2csnet_gaze360.onnx
L2CS-Net ONNX session ready.
  Inputs:  ['face_input']
  Outputs: ['pitch_pred', 'yaw_pred']


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# L2CS-Net ONNX INFERENCE (for Snowflake — no PyTorch)
# ══════════════════════════════════════════════════════════════════════════════

_L2CS_SOFTMAX_IDX = np.arange(-99, 100, dtype=np.float32)  # 199 bins
_L2CS_IMG_SIZE    = 448

def l2cs_onnx_predict(face_bgr, ort_sess):
    """
    L2CS-Net via ONNX runtime. Returns (yaw_deg, pitch_deg) or (None, None).
    Uses the same softmax expectation decoding as the original paper.
    """
    if face_bgr is None or ort_sess is None: return None, None
    try:
        img = cv2.resize(face_bgr, (_L2CS_IMG_SIZE,_L2CS_IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        img = (img - _NORM_MEAN) / _NORM_STD
        inp = img.transpose(2,0,1)[np.newaxis].astype(np.float32)
        out = ort_sess.run(None, {ort_sess.get_inputs()[0].name: inp})
        # out[0] = pitch logits (1, 198 or 199), out[1] = yaw logits
        pitch_logits = out[0][0]; yaw_logits = out[1][0]
        pitch_soft = np.exp(pitch_logits) / np.sum(np.exp(pitch_logits))
        yaw_soft   = np.exp(yaw_logits)   / np.sum(np.exp(yaw_logits))
        bins = np.arange(len(pitch_soft)) * (198.0/(len(pitch_soft)-1)) - 99.0
        pitch_deg = float(np.sum(bins * pitch_soft))
        yaw_deg   = float(np.sum(bins * yaw_soft))
        return yaw_deg, pitch_deg
    except Exception:
        return None, None

# Quick test
dummy_face = np.zeros((224,224,3),dtype=np.uint8)
y_onnx, p_onnx = l2cs_onnx_predict(dummy_face, _l2cs_ort_sess)
print(f"L2CS ONNX test: yaw={y_onnx:.1f}° pitch={p_onnx:.1f}°")

L2CS ONNX test: yaw=-27.0° pitch=-44.5°


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# GAZE FUSION + ZONE MAPPING
# ══════════════════════════════════════════════════════════════════════════════

def fuse_gaze(yaw_whe, pitch_whe, yaw_l2c, pitch_l2c, face_yaw_approx=0.0):
    """
    Fuse WHENet and L2CS-Net gaze predictions.
    
    At high face yaw (driver turned away from camera), L2CS-Net becomes
    unreliable because eye crops are partially occluded. In that case,
    fall back to WHENet head pose only.
    
    Returns (fused_yaw, fused_pitch, source_label).
    """
    whe_ok = yaw_whe is not None and pitch_whe is not None
    l2c_ok = yaw_l2c is not None and pitch_l2c is not None
    
    if not whe_ok and not l2c_ok:
        return None, None, "none"
    if not l2c_ok or abs(face_yaw_approx) > HIGH_YAW_FALLBACK_DEG:
        return yaw_whe, pitch_whe, "whenet_only"
    if not whe_ok:
        return yaw_l2c, pitch_l2c, "l2cs_only"
    
    # Weighted fusion
    fused_yaw   = FUSION_W_WHENET*yaw_whe   + FUSION_W_L2CS*yaw_l2c
    fused_pitch = FUSION_W_WHENET*pitch_whe + FUSION_W_L2CS*pitch_l2c
    return float(fused_yaw), float(fused_pitch), "fused"


def pose_to_zone(yaw_deg: float, pitch_deg: float) -> Tuple[str, int]:
    """
    Map world-frame gaze angles (after camera offset correction) to zone.
    Returns (zone_name, offroad_binary).
    
    Zone geometry (LISA reference):
      Forward:     |yaw|≤15° AND |pitch|≤15°
      Right Mirror: yaw>20°
      Left Mirror:  yaw<-20°
      Shoulder:    |yaw|>35° (more extreme lateral)
      Speedometer:  pitch<-15° AND |yaw|≤15°
      Lap:          pitch<-25° (strongly down)
      Radio:        pitch<-15° AND |yaw|>15° (down+side)
      Rearview:     pitch>15° AND |yaw|≤20° (looking up)
    """
    t = ZONE_THRESHOLDS
    abs_yaw = abs(yaw_deg)
    
    if abs_yaw > t["yaw_shoulder"]:    return "Shoulder",     1
    if yaw_deg > t["yaw_mirror"]:      return "Right Mirror",  1
    if yaw_deg < -t["yaw_mirror"]:     return "Left Mirror",   1
    if pitch_deg < t["pitch_lap"]:     return "Lap",           1
    if pitch_deg < t["pitch_down"] and abs_yaw > t["yaw_radio"]:  return "Radio", 1
    if pitch_deg < t["pitch_down"]:    return "Speedometer",   1
    if pitch_deg > t["pitch_rearview"] and abs_yaw < t["yaw_mirror"]: return "Rearview", 1
    if abs_yaw <= t["yaw_forward"] and abs(pitch_deg) <= t["pitch_forward"]: return "Forward", 0
    return "Forward", 0  # borderline → forward (safe default)


class GazeCalibration:
    """
    Per-session A-pillar camera offset calibration.
    
    During first CALIB_FRAMES frames (driver looks forward at road):
    - Collect fused yaw/pitch predictions
    - The median = camera offset from world-forward
    - Subtract this from all subsequent predictions
    
    This handles ANY camera angle from 0° (frontal) to 45° (A-pillar)
    without needing the user to specify the angle.
    """
    def __init__(self, calib_frames=25):   # ~5s at 5fps
        self._target  = calib_frames
        self._yaws    = []
        self._pitches = []
        self.yaw_offset   = 0.0
        self.pitch_offset = 0.0
        self.complete     = False

    def update(self, yaw, pitch):
        if self.complete or yaw is None: return
        self._yaws.append(yaw); self._pitches.append(pitch)
        if len(self._yaws) >= self._target:
            self.yaw_offset   = float(np.median(self._yaws))
            self.pitch_offset = float(np.median(self._pitches))
            self.complete     = True
            print(f"[GazeCalib] Complete. "
                  f"Camera offset: yaw={self.yaw_offset:.1f}° pitch={self.pitch_offset:.1f}°")

    def correct(self, yaw, pitch):
        if yaw is None: return yaw, pitch
        return yaw - self.yaw_offset, pitch - self.pitch_offset

    @property
    def progress_pct(self):
        return min(100, int(100 * len(self._yaws) / self._target))


class TemporalGazeSmooth:
    """Rolling average over last N fused gaze angles."""
    def __init__(self, n=3):
        self._n   = n
        self._yaws   = []
        self._pitches= []

    def update(self, yaw, pitch):
        if yaw is None: return None, None
        self._yaws.append(yaw); self._pitches.append(pitch)
        if len(self._yaws) > self._n: self._yaws.pop(0); self._pitches.pop(0)
        return float(np.mean(self._yaws)), float(np.mean(self._pitches))

print("Gaze fusion, zone mapping, calibration defined.")

Gaze fusion, zone mapping, calibration defined.


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# PER-FRAME GAZE PREDICTION PIPELINE
# This is the function that runs in the Streamlit app.
# ══════════════════════════════════════════════════════════════════════════════

def predict_gaze_zone(frame_bgr: np.ndarray,
                       calib: GazeCalibration,
                       smoother: TemporalGazeSmooth,
                       whenet_sess: ort.InferenceSession,
                       l2cs_sess:   ort.InferenceSession) -> dict:
    """
    Full gaze zone prediction for one frame.
    Returns dict with zone_pred, offroad_pred, yaw, pitch, confidence, source.
    """
    result = {
        "zone_pred":   "Unknown", "offroad_pred": 0,
        "offroad_prob": 0.0, "confidence": 0.0,
        "yaw_raw":None, "pitch_raw":None, "yaw_corrected":None, "pitch_corrected":None,
        "source":"none", "calib_complete": calib.complete,
        "risk_group_pred": "Unknown",
        "attn_label": "Calibrating…" if not calib.complete else "On-road",
    }

    # Step 1: Face detection
    face_224, face_yaw_approx, det = yunet_detect(frame_bgr)
    if face_224 is None: return result

    # Step 2: WHENet head pose
    yaw_whe, pitch_whe, _ = whenet_predict(face_224)

    # Step 3: L2CS-Net gaze
    yaw_l2c, pitch_l2c = l2cs_onnx_predict(face_224, l2cs_sess)

    # Step 4: Fusion
    fused_yaw, fused_pitch, source = fuse_gaze(
        yaw_whe, pitch_whe, yaw_l2c, pitch_l2c, face_yaw_approx)
    if fused_yaw is None: return result
    result["yaw_raw"] = fused_yaw
    result["pitch_raw"] = fused_pitch
    result["source"] = source

    # Step 5: Calibration update / correction
    calib.update(fused_yaw, fused_pitch)
    corrected_yaw, corrected_pitch = calib.correct(fused_yaw, fused_pitch)
    result["yaw_corrected"]   = corrected_yaw
    result["pitch_corrected"] = corrected_pitch

    # Step 6: Temporal smoothing
    smooth_yaw, smooth_pitch = smoother.update(corrected_yaw, corrected_pitch)

    # Step 7: Zone mapping
    zone, offroad = pose_to_zone(smooth_yaw, smooth_pitch)
    
    # Confidence: higher when both models agree
    if source == "fused" and yaw_whe is not None and yaw_l2c is not None:
        angle_diff = abs(yaw_whe - yaw_l2c) + abs(pitch_whe - pitch_l2c)
        confidence = float(np.clip(1.0 - angle_diff/60.0, 0.3, 0.95))
    elif source == "none":
        confidence = 0.0
    else:
        confidence = 0.60

    RISK = {"Forward":"Safe","Rearview":"Safe",
            "Left Mirror":"LowRisk","Right Mirror":"LowRisk",
            "Lap":"HighRisk","Radio":"HighRisk",
            "Shoulder":"HighRisk","Speedometer":"HighRisk"}
    result.update({
        "zone_pred":        zone,
        "offroad_pred":     offroad,
        "offroad_prob":     float(offroad) * confidence,
        "confidence":       confidence,
        "risk_group_pred":  RISK.get(zone, "Unknown"),
        "attn_label":       "On-road" if offroad==0 else "Off-road",
        "zone_top2":        zone,   # single source, no ensemble rank
        "calib_complete":   calib.complete,
    })
    return result

print("predict_gaze_zone() defined.")

predict_gaze_zone() defined.


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# EVALUATE ON NTHU TEST SET
#
# Ground truth: NTHU-DDD pseudo-labels (from solvePnP in fine-tune v2).
# These are imperfect (~10-15° error) but they're the best we have for NTHU.
# We compare our WHENet+L2CS vs the pseudo-label ground truth.
# Also compare against EfficientNet fine-tune v2 results.
# ══════════════════════════════════════════════════════════════════════════════

PSEUDO_CACHE_PATH = "/kaggle/input/datasets/suyashpokle2/b-pseudo-label-cache/b_pseudo_label_cache.json"

def load_pseudo_labels(cache_path):
    """Load pseudo-labels from fine-tune v2 cache."""
    if not os.path.exists(cache_path):
        raise FileNotFoundError(
            f"Pseudo-label cache not found: {cache_path}\n"
            "Run mod-b-finetune-v2.ipynb first to generate the cache.")
    with open(cache_path) as f: cache = json.load(f)
    print(f"[Pseudo] Loaded {len(cache)} entries.")
    return cache

def load_nthu_test_paths(manifest_root, fold_id, data_dir):
    """Load NTHU fold test CSV and return image paths."""
    csv_path = os.path.join(manifest_root, "nthu", f"fold_{fold_id}", "test.csv")
    df = pd.read_csv(csv_path)
    paths = []
    for p in df["path"].tolist():
        p = str(p).strip().replace("\\","/")
        full = os.path.join(data_dir, p)
        if os.path.exists(full): paths.append(full)
        elif os.path.exists(p): paths.append(p)
    print(f"[NTHU fold {fold_id} test] {len(paths)} frames")
    return paths


def evaluate_on_nthu_test(max_frames=2000, fold_id=1):
    """
    Run WHENet+L2CS on NTHU test frames and compare with pseudo-labels.
    Uses first max_frames frames for speed.
    """
    print(f"Loading pseudo-label cache…")
    pseudo = load_pseudo_labels(PSEUDO_CACHE_PATH)
    
    print(f"Loading NTHU test paths…")
    test_paths = load_nthu_test_paths(MANIFEST_ROOT, fold_id, NTHU_DATA_DIR)
    test_paths = test_paths[:max_frames]
    
    # Initialize calibration and smoother for this evaluation
    # Note: for eval we DON'T calibrate (no a priori knowledge of forward)
    # We use zero offset — this shows raw model performance
    eval_calib   = GazeCalibration(calib_frames=999999)  # never completes → zero offset
    eval_calib.complete = True  # skip calibration
    eval_smoother = TemporalGazeSmooth(n=TEMPORAL_SMOOTH_FRAMES)

    y_true, y_pred = [], []
    y_true_off, y_pred_off = [], []
    forward_idx = ZONE_TO_IDX["Forward"]
    n_no_face = 0; n_no_pseudo = 0

    for i, img_path in enumerate(test_paths):
        if (i+1)%200==0: print(f"  {i+1}/{len(test_paths)}…")

        pseudo_entry = pseudo.get(img_path, {})
        gt_zone8   = pseudo_entry.get("zone8", -1)
        gt_offroad = pseudo_entry.get("offroad", -1)
        if gt_zone8 < 0:
            n_no_pseudo += 1; continue

        frame = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if frame is None: n_no_face += 1; continue

        res = predict_gaze_zone(frame, eval_calib, eval_smoother,
                                 _whenet_sess, _l2cs_ort_sess)
        if res["zone_pred"] == "Unknown": n_no_face += 1; continue

        pred_zone8 = ZONE_TO_IDX.get(res["zone_pred"], -1)
        if pred_zone8 < 0: continue

        y_true.append(gt_zone8); y_pred.append(pred_zone8)
        y_true_off.append(max(0, gt_offroad)); y_pred_off.append(res["offroad_pred"])

    print(f"\nResults ({len(y_true)} frames evaluated):")
    print(f"  No face detected:  {n_no_face}")
    print(f"  No pseudo-label:   {n_no_pseudo}")

    if not y_true:
        print("No frames evaluated — check paths."); return {}

    # Per-zone F1
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(ZONE_CLASSES))))
    zone_f1s = []
    print("\nPer-zone F1:")
    for i, zname in enumerate(ZONE_CLASSES):
        tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp
        d=2*tp+fp+fn
        f1=float(2*tp/d) if d>0 else 0.0
        zone_f1s.append(f1)
        support = int(cm[i,:].sum())
        print(f"  {zname:15s}: F1={f1:.4f}  support={support}")

    macro_f1 = float(np.mean(zone_f1s))
    fwd_f1   = zone_f1s[forward_idx]
    off_f1   = float(f1_score(y_true_off, y_pred_off, zero_division=0))

    print(f"\n{'='*50}")
    print(f"zone_macro_f1 = {macro_f1:.4f}")
    print(f"forward_f1    = {fwd_f1:.4f}  (target: >0.60)")
    print(f"offroad_f1    = {off_f1:.4f}  (target: >0.90)")
    print(f"{'='*50}")

    return {"macro_f1":macro_f1,"forward_f1":fwd_f1,"offroad_f1":off_f1,
            "zone_f1s":dict(zip(ZONE_CLASSES,zone_f1s)),"n_frames":len(y_true)}

print("Evaluation function defined.")

Evaluation function defined.


In [15]:
# # ══════════════════════════════════════════════════════════════════════════════
# # RUN EVALUATION
# # ══════════════════════════════════════════════════════════════════════════════

# print("Starting evaluation (2000 NTHU test frames)…")
# eval_results = evaluate_on_nthu_test(max_frames=2000, fold_id=1)

# print("\n" + "="*60)
# print("COMPARISON: WHENet+L2CS-Net vs EfficientNet fine-tune v2")
# print("="*60)
# print(f"  EfficientNet v2:")
# print(f"    zone_macro_f1 = 0.196  forward_f1 = 0.108  offroad_f1 = 0.954")
# print(f"  WHENet + L2CS-Net:")
# print(f"    zone_macro_f1 = {eval_results.get('macro_f1',0):.3f}  "
#       f"forward_f1 = {eval_results.get('forward_f1',0):.3f}  "
#       f"offroad_f1 = {eval_results.get('offroad_f1',0):.3f}")

# fwd_improvement = (eval_results.get('forward_f1',0) - 0.108) / max(0.108,1e-9) * 100
# print(f"\nForward F1 improvement: {fwd_improvement:+.1f}%")

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# CALIBRATION IMPACT TEST
# Re-run with simulated A-pillar calibration offset
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_with_apillar_calibration(max_frames=1000, camera_yaw_offset=20.0):
    """
    Simulate A-pillar camera: apply a fixed yaw offset and measure impact.
    camera_yaw_offset: how many degrees the camera is rotated from driver's face centre.
    Positive = camera is to the right of the driver (India RHD A-pillar).
    """
    pseudo = load_pseudo_labels(PSEUDO_CACHE_PATH)
    test_paths = load_nthu_test_paths(MANIFEST_ROOT, 1, NTHU_DATA_DIR)[:max_frames]

    # Simulate calibration: tell the system offset is camera_yaw_offset degrees
    calib = GazeCalibration(calib_frames=1)
    calib.yaw_offset   = camera_yaw_offset
    calib.pitch_offset = 0.0
    calib.complete     = True

    smoother = TemporalGazeSmooth(n=3)
    y_true_fwd, y_pred_fwd = [], []
    y_true_off, y_pred_off = [], []

    for i, img_path in enumerate(test_paths):
        if (i+1)%200==0: print(f"  {i+1}/{len(test_paths)}…")
        pe = pseudo.get(img_path, {})
        gt_zone8  = pe.get("zone8",-1)
        gt_offrd  = pe.get("offroad",-1)
        if gt_zone8<0: continue
        frame = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if frame is None: continue
        res = predict_gaze_zone(frame, calib, smoother, _whenet_sess, _l2cs_ort_sess)
        if res["zone_pred"]=="Unknown": continue
        y_true_fwd.append(1 if gt_zone8==ZONE_TO_IDX["Forward"] else 0)
        y_pred_fwd.append(1 if res["zone_pred"]=="Forward" else 0)
        y_true_off.append(max(0,gt_offrd))
        y_pred_off.append(res["offroad_pred"])

    if not y_true_fwd: return
    fwd_f1 = float(f1_score(y_true_fwd, y_pred_fwd, zero_division=0))
    off_f1 = float(f1_score(y_true_off, y_pred_off, zero_division=0))
    print(f"\nWith A-pillar offset={camera_yaw_offset:.0f}°:")
    print(f"  forward_f1 = {fwd_f1:.4f}")
    print(f"  offroad_f1 = {off_f1:.4f}")

# # Test multiple offsets to find the optimal
# for offset in [0.0, 10.0, 15.0, 20.0, 25.0, 30.0]:
#     evaluate_with_apillar_calibration(max_frames=800, camera_yaw_offset=offset)

# Only test small offsets for NTHU (frontal camera)
for offset in [0.0, 2.5, 5.0, 7.5, 10.0]:
    evaluate_with_apillar_calibration(max_frames=800, camera_yaw_offset=offset)

[Pseudo] Loaded 66521 entries.
[NTHU fold 1 test] 19016 frames
  200/800…
  400/800…
  600/800…
  800/800…

With A-pillar offset=0°:
  forward_f1 = 0.2190
  offroad_f1 = 0.5771
[Pseudo] Loaded 66521 entries.
[NTHU fold 1 test] 19016 frames
  200/800…
  400/800…
  600/800…
  800/800…

With A-pillar offset=2°:
  forward_f1 = 0.2208
  offroad_f1 = 0.6630
[Pseudo] Loaded 66521 entries.
[NTHU fold 1 test] 19016 frames
  200/800…
  400/800…
  600/800…
  800/800…

With A-pillar offset=5°:
  forward_f1 = 0.2096
  offroad_f1 = 0.8640
[Pseudo] Loaded 66521 entries.
[NTHU fold 1 test] 19016 frames
  200/800…
  400/800…
  600/800…
  800/800…

With A-pillar offset=8°:
  forward_f1 = 0.0325
  offroad_f1 = 0.9172
[Pseudo] Loaded 66521 entries.
[NTHU fold 1 test] 19016 frames
  200/800…
  400/800…
  600/800…
  800/800…

With A-pillar offset=10°:
  forward_f1 = 0.0000
  offroad_f1 = 0.9227


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE GAZE PREDICTOR MODULE (for Streamlit app)
# This is the standalone Python module the Streamlit app imports.
# ══════════════════════════════════════════════════════════════════════════════

gaze_predictor_code = '''
"""
gaze_zone_predictor.py
WHENet + L2CS-Net fused gaze zone prediction.
Replaces EfficientNet Module B for gaze zone classification.
Works for frontal cameras (0°) and A-pillar cameras (15-30°).
"""
import os, json
import cv2
import numpy as np
import onnxruntime as ort

ZONE_CLASSES = ["Forward","Lap","Left Mirror","Radio",
                "Rearview","Right Mirror","Shoulder","Speedometer"]
ZONE_TO_IDX  = {z:i for i,z in enumerate(ZONE_CLASSES)}
RISK_MAP = {"Forward":"Safe","Rearview":"Safe",
            "Left Mirror":"LowRisk","Right Mirror":"LowRisk",
            "Lap":"HighRisk","Radio":"HighRisk",
            "Shoulder":"HighRisk","Speedometer":"HighRisk"}

ZONE_THR = {
    "yaw_shoulder":35.0,"yaw_mirror":20.0,
    "pitch_lap":-25.0,"pitch_down":-15.0,"yaw_radio":15.0,
    "pitch_rearview":15.0,"yaw_forward":15.0,"pitch_forward":15.0,
}
_NORM_MEAN = np.array([0.485,0.456,0.406],dtype=np.float32)
_NORM_STD  = np.array([0.229,0.224,0.225],dtype=np.float32)


class GazeZonePredictor:
    """Drop-in replacement for Module B. Requires WHENet + L2CS ONNX files."""

    def __init__(self, whenet_path, l2cs_path, calib_frames=25):
        opts = ort.SessionOptions()
        opts.intra_op_num_threads = 2
        providers = ["CPUExecutionProvider"]
        self._whe = ort.InferenceSession(whenet_path, opts, providers=providers)
        self._l2c = ort.InferenceSession(l2cs_path,   opts, providers=providers)
        self._calib_target = calib_frames
        self.reset_session()

    def reset_session(self):
        """Call at the start of every new video/session."""
        self._calib_yaws    = []
        self._calib_pitches = []
        self._calib_complete= False
        self._yaw_offset    = 0.0
        self._pitch_offset  = 0.0
        self._smooth_yaws   = []
        self._smooth_pitches= []

    @property
    def calibration_complete(self): return self._calib_complete
    @property
    def calibration_pct(self):
        return min(100, int(100*len(self._calib_yaws)/self._calib_target))

    def _whenet(self, face_224):
        img = cv2.cvtColor(face_224,cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        img = (img-_NORM_MEAN)/_NORM_STD
        inp = img.transpose(2,0,1)[np.newaxis].astype(np.float32)
        out = self._whe.run(None,{self._whe.get_inputs()[0].name:inp})
        if len(out)==3: return float(out[0].squeeze()),float(out[1].squeeze())
        angles=out[0].squeeze(); return float(angles[0]),float(angles[1])

    def _l2cs(self, face_bgr):
        img = cv2.resize(face_bgr,(448,448))
        img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        img = (img-_NORM_MEAN)/_NORM_STD
        inp = img.transpose(2,0,1)[np.newaxis].astype(np.float32)
        out = self._l2c.run(None,{self._l2c.get_inputs()[0].name:inp})
        pitch_l=out[0][0]; yaw_l=out[1][0]
        bins = np.arange(len(pitch_l))*(198.0/(len(pitch_l)-1))-99.0
        def softmax_expect(logits):
            e=np.exp(logits-logits.max()); s=e/e.sum()
            return float(np.sum(bins[:len(logits)]*s))
        return softmax_expect(yaw_l), softmax_expect(pitch_l)

    def _zone_map(self, yaw, pitch):
        t=ZONE_THR; ay=abs(yaw)
        if ay>t["yaw_shoulder"]:   return "Shoulder",1
        if yaw>t["yaw_mirror"]:    return "Right Mirror",1
        if yaw<-t["yaw_mirror"]:   return "Left Mirror",1
        if pitch<t["pitch_lap"]:   return "Lap",1
        if pitch<t["pitch_down"] and ay>t["yaw_radio"]: return "Radio",1
        if pitch<t["pitch_down"]:  return "Speedometer",1
        if pitch>t["pitch_rearview"] and ay<t["yaw_mirror"]: return "Rearview",1
        if ay<=t["yaw_forward"] and abs(pitch)<=t["pitch_forward"]: return "Forward",0
        return "Forward",0

    def predict(self, face_224_bgr, face_yaw_approx=0.0):
        """
        Predict gaze zone from 224×224 face crop.
        Returns dict compatible with existing Streamlit app infer_b() output.
        """
        default = {"zone_pred":"Unknown","offroad_pred":0,"offroad_prob":0.0,
                   "confidence":0.0,"attn_label":"Calibrating…" if not self._calib_complete else "On-road",
                   "risk_group_pred":"Unknown","calib_pct":self.calibration_pct}
        if face_224_bgr is None: return default
        try:
            yaw_w, pitch_w = self._whenet(face_224_bgr)
            yaw_l, pitch_l = self._l2cs(face_224_bgr)
            # Fusion
            use_l2c = abs(face_yaw_approx) <= 35.0
            if use_l2c:
                fy = 0.35*yaw_w + 0.65*yaw_l
                fp = 0.35*pitch_w + 0.65*pitch_l
                conf = max(0.3, 1.0 - (abs(yaw_w-yaw_l)+abs(pitch_w-pitch_l))/60.0)
                src = "fused"
            else:
                fy,fp,conf,src = yaw_w,pitch_w,0.60,"whenet_only"
            # Calibration
            if not self._calib_complete:
                self._calib_yaws.append(fy); self._calib_pitches.append(fp)
                if len(self._calib_yaws)>=self._calib_target:
                    self._yaw_offset   = float(np.median(self._calib_yaws))
                    self._pitch_offset = float(np.median(self._calib_pitches))
                    self._calib_complete = True
            fy -= self._yaw_offset; fp -= self._pitch_offset
            # Temporal smooth
            self._smooth_yaws.append(fy); self._smooth_pitches.append(fp)
            if len(self._smooth_yaws)>3: self._smooth_yaws.pop(0); self._smooth_pitches.pop(0)
            sy = float(np.mean(self._smooth_yaws))
            sp = float(np.mean(self._smooth_pitches))
            # Zone
            zone, offroad = self._zone_map(sy, sp)
            attn = "On-road" if offroad==0 else "Off-road"
            return {"zone_pred":zone,"offroad_pred":offroad,
                    "offroad_prob":float(offroad)*conf,
                    "confidence":float(np.clip(conf,0,1)),
                    "attn_label":attn,
                    "risk_group_pred":RISK_MAP.get(zone,"Unknown"),
                    "yaw_corrected":float(sy),"pitch_corrected":float(sp),
                    "source":src,"calib_pct":self.calibration_pct,
                    "zone_top2":zone,
                    **{f"zone_prob_{z.lower().replace(\' \')\'_\'}": \
                        1.0 if z==zone else 0.0 for z in ZONE_CLASSES}}
        except Exception as e:
            return default
'''

gaze_py_path = os.path.join(OUTPUT_DIR, "gaze_zone_predictor.py")
with open(gaze_py_path,"w") as f:
    f.write(gaze_predictor_code)
print(f"Saved: {gaze_py_path}")

Saved: /kaggle/working/gaze_v3/gaze_zone_predictor.py


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE META + COPY OUTPUTS
# ══════════════════════════════════════════════════════════════════════════════

meta = {
    "version":        "b_gaze_v3_whenet_l2cs_fusion",
    "approach":       "WHENet (head pose) + L2CS-Net (gaze) fused, no ML training",
    "zone_classes":   ZONE_CLASSES,
    "zone_thresholds": ZONE_THRESHOLDS,
    "fusion_w_whenet": FUSION_W_WHENET,
    "fusion_w_l2cs":   FUSION_W_L2CS,
    "temporal_smooth_frames": TEMPORAL_SMOOTH_FRAMES,
    "calib_frames":    25,
    "camera_compat":  "0-45 degree yaw (frontal + A-pillar)",
    "whenet_input":   "(1,3,224,224) RGB ImageNet-normalised",
    "l2cs_input":     "(1,3,448,448) RGB ImageNet-normalised",
    "note":           ("Upload whenet_1x3x224x224_prepost.onnx and "
                       "l2csnet_gaze360.onnx to Snowflake stage. "
                       "In Streamlit, replace load_b() with GazeZonePredictor."),
    "eval_results":   eval_results,
}

meta_path = os.path.join(OUTPUT_DIR, "b_gaze_v3_meta.json")
with open(meta_path,"w") as f: json.dump(meta, f, indent=2)
print(f"Meta saved: {meta_path}")

# Copy to /kaggle/working root for download
import shutil
for src, dst in [
    (L2CS_ONNX_OUT,  "/kaggle/working/l2csnet_gaze360.onnx"),
    (meta_path,      "/kaggle/working/b_gaze_v3_meta.json"),
    (gaze_py_path,   "/kaggle/working/gaze_zone_predictor.py"),
]:
    if not os.path.exists(src):
        continue

    if os.path.abspath(src) == os.path.abspath(dst):
        print(f"Already at destination, skipped: {src}")
        continue

    shutil.copy2(src, dst)
    print(f"Copied → {dst}")
    #if os.path.exists(src): shutil.copy2(src,dst); print(f"Copied → {dst}")

print("\nDownload from /kaggle/working:")
print("  l2csnet_gaze360.onnx")
print("  b_gaze_v3_meta.json")
print("  gaze_zone_predictor.py")
print("\nUpload to Snowflake:")
print("  PUT file:///path/l2csnet_gaze360.onnx @DEMO_DB.PUBLIC.DRIVER_SAFETY_MODELS AUTO_COMPRESS=FALSE;")
print("  (whenet_1x3x224x224_prepost.onnx already uploaded ✓)")

NameError: name 'eval_results' is not defined